# Data preprocessing

## Introduction

Notebooks 2,3 and 4 will train progressively more advanced encoder-decoder translation modelsBefore we can train any translation models, we need to prepare the data. The preprocessing pipeline includes the following:

1. Pull the OPUS-100 English-French corpus<sup>1</sup> from Hugging Face: [Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100)
2. Tokenize the translation pairs with the cognate MarianTokenizer tokenizer: [Helsinki-NLP/opus-mt-en-fr](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr)
3. Apply a length threshold to the translation pairs (100 tokens)
4. Reserve 1000 randomly sampled translation pairs for testing
5. Tokenize the translation pairs with padding up to max length. This step ensures that all of the sequences have the same length.
6. Add a 'beginning of sequence' token (BOS) to the decoder input.
7. Add an 'end of sequence' token (EOS) to the decoder target.
8. Split the tokenized and padded translation pairs into training and validation sets.
9. Use Python generators to create TensorFlow datasets for the training and validation data.

The purpose of the BOS and EOS tokens will become clear when we look at the encoder-decoder architecture in notebook 02.

## References

1. Zhang, B., Williams, P., Titov, I., & Sennrich, R. (2020). **Improving massively multilingual neural machine translation and zero-shot translation.** *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics (ACL).* https://arxiv.org/abs/2004.11867

## 1. Notebook setup
### 1.1. Imports

In [1]:
import pickle
from dotenv import load_dotenv

import numpy as np
import tensorflow as tf

from datasets import load_dataset
from huggingface_hub.utils import disable_progress_bars
from sklearn.model_selection import train_test_split
from transformers import MarianTokenizer

import configuration as config

# Initialization
np.random.seed(315)
tf.random.set_seed(315)

### 1.2. Environment configuration

In [2]:
# Disable Jupyter widgets to prevent rendering issues after reopening notebook
disable_progress_bars()

# Environment variables
load_dotenv(config.PROJECT_ROOT / '.env')  # For HuggingFace upload (HF_TOKEN, HF_REPO_ID)

True

## 2. Load prepare assets

### 2.1. Tokenizer

In [3]:
# Load pre-trained subword tokenizer MarianTokenizer (SentencePiece),
# specifically designed for the Helsinki-NLP translation models
tokenizer = MarianTokenizer.from_pretrained(
    config.GLOBAL_CONFIG['tokenizer'],
    cache_dir=config.MODEL_DIRECTORY
)

print(f'Tokenizer vocabulary size: {tokenizer.vocab_size}')
print(f'Special tokens: {tokenizer.special_tokens_map}')

# Example of subword tokenization
example = 'The neural network learned representations'
tokens = tokenizer.tokenize(example)

print(f'\nExample tokenization:')
print(f'  Input: "{example}"')
print(f'  Tokens: {tokens}')

Tokenizer vocabulary size: 59514
Special tokens: {'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}

Example tokenization:
  Input: "The neural network learned representations"
  Tokens: ['▁The', '▁ne', 'ural', '▁network', '▁learned', '▁representations']


### 2.2. Dataset

In [4]:
# Load OPUS-100 English-French translation dataset
dataset = load_dataset(
    config.GLOBAL_CONFIG['dataset'],
    'en-fr',
    cache_dir=config.DATA_DIRECTORY
)

# Extract translation pairs and filter by token length
pairs = []

for item in dataset['train']:

    en_text = item['translation']['en'].strip()
    fr_text = item['translation']['fr'].strip()
    
    # Check token length using tokenize() to avoid truncation warnings
    en_tokens = tokenizer.tokenize(en_text)
    fr_tokens = tokenizer.tokenize(fr_text)
    
    if len(en_tokens) <= config.GLOBAL_CONFIG['max_seq_length'] and len(fr_tokens) <= config.GLOBAL_CONFIG['max_seq_length']:
        pairs.append((en_text, fr_text))
    
    # Limit dataset size
    if len(pairs) >= config.GLOBAL_CONFIG['num_train_samples']:
        break

print(f'Loaded {len(pairs)} translation pairs')
print(f'\nSample pairs:')

for en, fr in pairs[1:5]:
    print()
    print(f'  EN: {en}')
    print(f'  FR: {fr}')

Loaded 2000 translation pairs

Sample pairs:

  EN: This Regulation shall enter into force on the seventh day following its publication in the Official Journal of the European Union.
  FR: Le présent règlement entre en vigueur le septième jour suivant celui de sa publication au Journal officiel de l'Union européenne.

  EN: Hello, what's that?
  FR: Qu'est-ce que c'est que ça ?

  EN: And then I will teach you everything i know.
  FR: Et alors, je t'apprendrai tout ce que je sais.

  EN: Did you find something?
  FR: Par ici !


### 2.3. Reserve test set

In [5]:
# Reserve n random pairs for BLEU evaluation and final test set
test_pairs = np.random.choice(
    len(pairs),
    size=config.GLOBAL_CONFIG['test_set_size'],
    replace=False
)

test_pairs = [pairs[i] for i in test_pairs]

# Remove the test pairs from the original dataset
pairs = [pair for pair in pairs if pair not in test_pairs]

print(f'Training+validation set: {len(pairs)} pairs')
print(f'Test set: {len(test_pairs)} pairs')
print(f'Test set is: {type(test_pairs)}')

Training+validation set: 987 pairs
Test set: 1000 pairs
Test set is: <class 'list'>


## 3. Tokenize training set with padding

In [6]:
# Set EOS, BOS, and PAD token IDs
eos_id, bos_id = tokenizer.eos_token_id, tokenizer.eos_token_id
pad_id = tokenizer.pad_token_id

In [7]:
# Tokenize source (English - index 0) sentences
encoder_inputs = tokenizer(
    [pair[0] for pair in pairs],
    padding='max_length',
    truncation=True,
    max_length=config.GLOBAL_CONFIG['max_encoder_length'],
    return_tensors='np'
)

# Tokenize target (French - index 1) sentences
decoder_inputs = tokenizer(
    [pair[1] for pair in pairs],
    padding='max_length',
    truncation=True,
    max_length=config.GLOBAL_CONFIG['max_decoder_length'],
    return_tensors='np'
)

In [8]:
# Prepare encoder data
encoder_input_data = encoder_inputs['input_ids']

# Prepare decoder data
raw_decoder_tokens = decoder_inputs['input_ids']

# Target is exactly the tokenizer output
decoder_target_data = raw_decoder_tokens.copy()

# Build Input: Shift right by 1 and prepend BOS
decoder_input_data = np.full_like(raw_decoder_tokens, pad_id)
decoder_input_data[:, 0] = bos_id
decoder_input_data[:, 1:] = raw_decoder_tokens[:, :-1]

# Model dimensions
num_samples = len(pairs)
num_tokens = tokenizer.vocab_size  # Same vocab for encoder and decoder

# FIX THE BURIED EOS: Find where EOS shifted to in the input, and turn it into PAD
# (Because the model shouldn't see EOS in the input context)
eos_indices = np.argwhere(decoder_input_data == eos_id)

for row, col in eos_indices:
    if col > 0: # Ignore the BOS token we just put at index 0
        decoder_input_data[row, col] = pad_id

print(f'Vocabulary size: {num_tokens}')
print(f'Max encoder length: {config.GLOBAL_CONFIG["max_encoder_length"]}')
print(f'Max decoder length: {config.GLOBAL_CONFIG["max_decoder_length"]}')
print(f'Training samples: {num_samples}')
print()
print(f'Encoder input shape: {encoder_input_data.shape}')
print(f'Decoder input shape: {decoder_input_data.shape}')
print(f'Decoder target shape: {decoder_target_data.shape}')
print()
print(f'Example encoder input: {tokenizer.decode(encoder_input_data[0], skip_special_tokens=False)}\n')
print(f'Example decoder input: {tokenizer.decode(decoder_input_data[0], skip_special_tokens=False)}\n')
print(f'Example decoder target: {tokenizer.decode(decoder_target_data[0], skip_special_tokens=False)}')

Vocabulary size: 59514
Max encoder length: 101
Max decoder length: 101
Training samples: 987

Encoder input shape: (987, 101)
Decoder input shape: (987, 101)
Decoder target shape: (987, 101)

Example encoder input: This Regulation shall enter into force on the seventh day following its publication in the Official Journal of the European Union.</s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>

Example decoder input: </s> Le présent règlement entre en vigueur le septième jour suivant celui de sa publication au Journal officiel de l'Union européenne.<pad> <pad> <pad> <pad> 

## 4. Training-validation split

In [9]:
# Split the data using an index array to keep all three datasets perfectly aligned
indices = range(len(encoder_input_data))

train_idx, val_idx = train_test_split(
    indices,
    test_size=config.GLOBAL_CONFIG['validation_split'],
    random_state=315
)

# Slice NumPy arrays into Train and Val sets
X_train_enc, X_val_enc = encoder_input_data[train_idx], encoder_input_data[val_idx]
X_train_dec, X_val_dec = decoder_input_data[train_idx], decoder_input_data[val_idx]
y_train, y_val         = decoder_target_data[train_idx], decoder_target_data[val_idx]

## 5. Create TensorFlow datasets

### 5.1. Python generators to stream data

In [10]:
# Create python generators that don't load the graph
def train_gen():
    for i in range(len(X_train_enc)):
        yield ({'encoder_input': X_train_enc[i], 'decoder_input': X_train_dec[i]}, y_train[i])

def val_gen():
    for i in range(len(X_val_enc)):
        yield ({'encoder_input': X_val_enc[i], 'decoder_input': X_val_dec[i]}, y_val[i])

### 5.2. Create data pipelines

In [ ]:
# Build the data pipelines
train_dataset = tf.data.Dataset.from_generator(
    train_gen, 
    output_signature=config.TF_OUTPUT_SIGNATURE
)

val_dataset = tf.data.Dataset.from_generator(
    val_gen, 
    output_signature=config.TF_OUTPUT_SIGNATURE
)

## 6. Save datasets

In [14]:
train_dataset.save(
    str(config.TRAIN_DATASET_PATH)
)

val_dataset.save(
    str(config.VAL_DATASET_PATH)
)

with open(config.TEST_DATASET_PATH, 'wb') as f:
    pickle.dump(test_pairs, f)